In [30]:
import os
import ast
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
import xgboost as xgb

# 1. Setup Base Data and Columns (Match this to your actual dataframe)
df = pd.read_csv('data/spectral_feature_data.csv')
base_spectral_columns = ['410', '435', '460', '485', ...]
target_cols = ['p2.N.low', 'p2.OC', 'p2.P', ...]


non_feature_cols = [col for col in df.columns if col.startswith('p4')]
feature_cols = [col for col in df.columns if col not in non_feature_cols]

base_spectral_columns = [col for col in feature_cols if not col.startswith("p")]
target_cols = [col for col in df.columns if col.startswith("p")]


config_map = {
    "Spectral Only": [False, False],
    "Spectral + pH": [True, False],
    "Spectral + EC": [False, True],
    "Spectral + pH + EC": [True, True]
}

# 2. Setup the Folder Structure
base_dir = "models"
model_types = ["plsr", "xgb", "lgbm", "rf"]

for m_type in model_types:
    os.makedirs(os.path.join(base_dir, m_type), exist_ok=True)
    print(f"Directory ready: {os.path.join(base_dir, m_type)}/")


print(target_cols, base_spectral_columns)

Directory ready: models\plsr/
Directory ready: models\xgb/
Directory ready: models\lgbm/
Directory ready: models\rf/
['p1.pH.index', 'p1.EC.ds_m', 'p1.Clay.wt_pct', 'p1.Sand.wt_pct', 'p1.Silt.wt_pct', 'p2.N.wt_pct', 'p2.Zn.mg_kg', 'p2.OC.wt_pct', 'p3.Fe.mg_kg', 'p3.K.mg_kg', 'p3.P.mg_kg', 'p3.S.wt_pct', 'p4.BD.g_cm3', 'p4.CEC.cmolc_kg', 'p4.CF.wt_pct', 'p4.WR_10kPa.wt_pct', 'p4.WR_1500kPa.wt_pct', 'p4.WR_33kPa.wt_pct'] ['410', '435', '460', '485', '510', '535', '560', '585', '610', '645', '680', '705', '730', '760', '810', '860', '900', '940']


In [31]:
# 3. Load your Winning Blueprints (Assuming you saved the best configs per model to CSVs)
# Note: Ensure you have these CSVs ready from your previous analysis steps.
blueprints = {
    "plsr": pd.read_csv("results/model_configs/PLSR_final_configs.csv"),
    "xgb": pd.read_csv("results/model_configs/XGBoost_final_configs.csv"),
    "lgbm": pd.read_csv("results/model_configs/LGBM_final_configs.csv"),
    "rf": pd.read_csv("results/model_configs/RF_final_configs.csv")
}


In [32]:

# 4. Master Loop: Train and Save
for m_type in model_types:
    print(f"\n========== BUILDING {m_type.upper()} MODELS ==========")
    blueprint_df = blueprints[m_type]
    
    for _, row in blueprint_df.iterrows():
        
        target = row['Feature']

        if m_type == "xgb":
            best_config = "Spectral + pH + EC"
        else:
            best_config = row['Config']
        

        # --- NEW: Conditional Parameter Parsing ---
        if m_type == "plsr":
            # PLSR only saved the integer under 'Best_Components'
            n_comp = int(row['Best_Components'])
            best_params = {'n_components': n_comp}
        else:
            # XGB, LGBM, and RF saved dictionary strings under 'Best_Params'
            best_params = ast.literal_eval(row['Best_Params']) if pd.notna(row['Best_Params']) else {}

        
        print(f"Training {m_type.upper()} for {target} using {best_config}...")

        # --- Reconstruct the exact dataset for this configuration ---
        prediction_columns = base_spectral_columns.copy()
        maskpH = pd.Series(True, index=df.index)
        maskEC = pd.Series(True, index=df.index)

        if config_map[best_config][0]: # pH is True
            prediction_columns.append("p1.pH.index")
            maskpH = df["p1.pH.index"].notna()
        if config_map[best_config][1]: # EC is True
            prediction_columns.append("p1.EC.ds_m")
            maskEC = df["p1.EC.ds_m"].notna()

        feature_mask = maskpH & maskEC
        X_custom = df[prediction_columns]
        
        target_mask = df[target].notna()
        final_mask = target_mask & feature_mask 
        
        y_clean = df.loc[final_mask, target]
        X_clean = X_custom.loc[final_mask]


        # --- NEW: DEBUGGING & SAFETY VALVE ---
        '''print(f"  -> Total rows in df: {len(df)}")
        print(f"  -> Rows with {target} (not NaN): {target_mask.sum()}")
        print(f"  -> Rows with pH (not NaN): {maskpH.sum()}")
        print(f"  -> Rows with EC (not NaN): {maskEC.sum()}")'''
        print(f"  -> Rows surviving all filters: {final_mask.sum()}")
        
        if final_mask.sum() == 0:
            print(f"⚠️ SKIPPING {m_type.upper()} for {target} - No data left after filtering!")
            continue

        # Use the EXACT same split so the Stacker can use the unseen X_test later
        X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

        # --- Initialize the specific Algorithm ---
        if m_type == "plsr":
            # PLSR expects 'n_components'
            engine = PLSRegression(**best_params)
        elif m_type == "xgb":
            engine = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, **best_params)
        elif m_type == "lgbm":
            engine = LGBMRegressor(random_state=42, verbose=-1, **best_params)
        elif m_type == "rf":
            engine = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)

        # --- Create the Production Pipeline ---
        # 1. Impute missing values automatically
        # 2. Run the algorithm
        model_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('model', engine)
        ])

        # --- Train on the Training Data ---
        # We wrap X_train in a DataFrame to prevent the feature name warnings
        X_train_df = pd.DataFrame(X_train, columns=prediction_columns)
        model_pipeline.fit(X_train_df, y_train)

        # --- Save the Pipeline ---
        # File name format: models/rf/p2.N.low_model.pkl
        file_path = os.path.join(base_dir, m_type, f"{target}_model.pkl")
        
        # We also save the 'prediction_columns' list so your main.py knows EXACTLY 
        # which columns this specific model expects to receive from the hardware.
        save_package = {
            'features_required': prediction_columns,
            'pipeline': model_pipeline
        }
        
        joblib.dump(save_package, file_path)
        print(f"  -> Saved to {file_path}")

print("\nAll winning models have been successfully serialized for production.")


========== BUILDING PLSR MODELS ==========
Training PLSR for p1.Clay.wt_pct using Spectral Only...
  -> Rows surviving all filters: 3915
  -> Saved to models\plsr\p1.Clay.wt_pct_model.pkl
Training PLSR for p1.EC.ds_m using Spectral + pH + EC...
  -> Rows surviving all filters: 21832
  -> Saved to models\plsr\p1.EC.ds_m_model.pkl
Training PLSR for p1.Sand.wt_pct using Spectral + pH + EC...
  -> Rows surviving all filters: 4306
  -> Saved to models\plsr\p1.Sand.wt_pct_model.pkl
Training PLSR for p1.Silt.wt_pct using Spectral + pH...
  -> Rows surviving all filters: 3755
  -> Saved to models\plsr\p1.Silt.wt_pct_model.pkl
Training PLSR for p1.pH.index using Spectral + pH + EC...
  -> Rows surviving all filters: 21832
  -> Saved to models\plsr\p1.pH.index_model.pkl
Training PLSR for p2.N.wt_pct using Spectral + pH + EC...
  -> Rows surviving all filters: 21832
  -> Saved to models\plsr\p2.N.wt_pct_model.pkl
Training PLSR for p2.OC.wt_pct using Spectral + pH + EC...
  -> Rows surviving all 